# Visual Novel Maker - 実装機能一覧

ブラウザベースのノベルゲーム制作ツール

## 📝 編集・制作機能

- **シーンの作成・編集・削除・順序変更** - ドラッグ&ドロップで直感的に管理
- **テキスト入力** - シーンごとに物語のセリフや地の文を記述
- **背景画像設定** - URL指定またはファイルアップロード両対応
- **フェード時間調整** - 次シーンへの切り替え時の暗転・復帰時間を秒単位で設定

## 🎭 キャラクター機能

- **キャラクター登録・管理** - キャラクター名と画像を登録
- **表情管理** - 1キャラクターに複数の表情を登録可能
- **シーン内での配置** - キャラの位置（X, Y）、サイズ、重ね順（Z）を調整
- **名前ラベル表示** - プレイ時にキャラの名前をラベル表示するかどうかを選択
- **キャラクターパレット** - 登録キャラの一覧表示・削除機能

## 🔊 音声機能

- **BGM設定** - ループ再生対応、前シーンから継続可能
- **効果音（SE）設定** - 1回のみ再生
- **音声試聴機能** - 編集中に音声を再生して確認
- **ファイルアップロード対応** - URL指定またはローカルファイルから選択

## 🎯 ストーリー制御機能

- **選択肢分岐** - 1シーンに複数の選択肢を設定可能
- **シーンジャンプ** - 任意のシーンへの遷移を指定（分岐・スキップに対応）
- **次シーン指定** - デフォルトでは次の番号のシーンへ、別シーンへの遷移も可能

## ☁️ データ管理機能

- **ローカルストレージ** - プロジェクトを自動保存
- **Supabaseクラウド同期** - Google認証でログイン、複数デバイス間でデータ同期
- **複数プロジェクト管理** - 複数の作品を同時に管理
- **JSON形式エクスポート** - 作品をJSONファイルで保存
- **JSON形式インポート** - 他の人が作った作品を読み込み
- **一括画像アップロード** - 複数の画像ファイルを一度にアップロード

## ✨ AIアシスト機能（試験的）

- **次シーンテキスト提案** - AIが物語の続きのテキストを提案
- **背景画像プロンプト生成** - シーンの説明から背景画像生成用のプロンプトを生成
- **カスタム指示実行** - ユーザーが指定した指示でAIが生成
- **軽量モデル** - Transformers.jsを使用したブラウザ内実行（外部API不要）

## 📖 その他機能

- **台本ビュー** - 全シーンのテキストを整形表示
  - 印刷対応（PDFダウンロード可）
  - テキストファイル保存
  - クリップボードコピー
  - シーン内編集可能

- **制作モード切り替え** - AIアシスト / スタンダードから選択

- **プレイ機能** - 作成したノベルをブラウザで再生
  - 選択肢クリック対応
  - キャラとBGMの表示・再生

- **PWA対応** - Service Worker実装でオフライン使用可能

- **レスポンシブUI** - PC・タブレット・モバイルに対応

## 🔐 認証機能

- **Google OAuth連携** - Googleアカウントでログイン
- **ユーザープロフィール表示** - ログイン後、ユーザー名とアバター表示
- **クラウド連携** - ログインユーザーのデータをクラウドに自動同期



## ❌ 未実装・拡張候補機能

- **高度なキャラクター演出**  (アニメーション、トランジションなど)
- **条件付き分岐・フラグ管理**  (親密度、フラグ判定)
- **セーブ／ロード機能**  (プレイヤー側の進行保存)
- **ストーリー分岐の可視化**  (フローチャート表示)
- **画像生成API連携**  (Stable Diffusion等)
- **ユーザーコラボレーション**  (複数人編集、共有)
- **バージョン履歴・復元**
- **作品共有・公開機能**
- **多言語対応**
- **アナリティクス統計**

上記はコード内から察せられる拡張候補で、まだ具体的な実装はありません。



## ⚡ パフォーマンス最適化

### 実装済み（v2.0以降）

- **AI初期化の再試行メカニズム** - 失敗時は自動で最大3回まで再試行（1秒間隔）
- **クラウド同期の信頼性向上** - save失敗時は自動で最大3回までリトライ（500ms間隔）
- **ユーザー情報キャッシング** - getUser()結果を30秒間キャッシュして、重複クエリを削減
- **ログイン時のデータ保護** - ログイン前に未保存データをクラウドに同期してからリロード
- **エラーハンドリング強化** - save()失敗時は例外をthrowして呼び元で処理可能に

### 予定中の最適化

- **大規模プロジェクト対応** - 1000シーン以上のデータを分割ロード
- **オフスクリーンレンダリング** - キャラクター画像の事前描画
- **インデックス導入** - シーン検索の高速化
- **デバウンス処理** - 連続保存を制限
- **メモリ管理** - 長時間使用での メモリリーク防止

## 🛠️ 実装詳細 - パフォーマンス改善内容

### 1. AI初期化の再試行メカニズム（ai.js）
```
変更内容：
- MAX_RETRIES = 3, RETRY_DELAY_MS = 1000ms を定数化
- init()失敗時は自動で3回まで再試行
- 各失敗時に1秒のディレイを挿入して、ネットワーク回復を待つ
- 効果：一時的なネットワーク障害での初期化失敗を防止
```

### 2. クラウド同期の再試行機能（storage.js）
```
変更内容：
- _retrySave()メソッドを新規追加
- save()失敗時は自動で最大3回リトライ（500ms間隔）
- CloudStorage.save()がthrowするようにして、エラーを適切に検出
- 効果：ネットワーク不安定性への耐性強化
```

### 3. ユーザー情報キャッシング（db.js）
```
変更内容：
- _userCache と _userCacheExpiry を追加
- getUser()結果を30秒間メモリキャッシュ
- signOut()及びonAuthStateChange時にキャッシュをクリア
- 効果：同一クエリの削減により、APIコール削減と応答速度向上
```

### 4. ログイン時のデータ自動保存（auth.js）
```
変更内容：
- SIGNED_IN イベント時に、ページリロード前に現在のプロジェクトデータを自動保存
- 500msの遅延後にリロード（Supabase同期完了を待つ）
- 効果：ログイン時の未保存データ损失防止
```